# Example script for Hackathon

Within each cycle of active learning, you can:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have a limited chances of making queries in total.

In [64]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

## 1. collect training data

Upload `sequence.fasta`, `train.csv`, and `test.csv` to the current runtime:

1. click the folder icon on the left

2. click the upload icon and upload the files to the current directory

In [35]:
with open('sequence.fasta', 'r') as f:
  data = f.readlines()

sequence_wt = data[1].strip()
sequence_wt

'MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP'

In [36]:
len(sequence_wt)

656

In [37]:
def get_mutated_sequence(mut, sequence_wt):
  wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]

  sequence = deepcopy(sequence_wt)

  return sequence[:pos]+mt+sequence[pos+1:]

In [38]:
df_train = pd.read_csv('train.csv')
df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_train

,mutant,DMS_score,sequence
0,M0Y,0.2730,YVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,M0W,0.2857,WVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,M0V,0.2153,VVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,M0T,0.3122,TVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,M0S,0.2180,SVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...,...
1135,P347D,0.3876,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1136,P347C,0.1837,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1137,P347A,0.4611,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1138,P347M,0.2412,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [39]:
df_test = pd.read_csv('test.csv')
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_test

,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [ ]:
# TODO: integrate the query data that you acquired each round into df_train

## 2. Train a prediction model

Here, we provided a linear regression model and used one-hot encoding to encode each variant. You would need to build your own model to achieve better performances.

Hint: you can perform cross-validation on the training set to evaluate your predictor before making predictions on the test set.

In [54]:
'''hyperparameters'''

seq_length = 656
seed = 0 # seed for splitting the validation set
val_ratio = 0.2 # proportion of validation set

In [55]:
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_idx = {aa: i for i, aa in enumerate(amino_acids)}

# def encode_mutation(mut, seq_length=656):
#     wt_aa, pos, mut_aa = mut[0], int(mut[1:-1]), mut[-1]
#     vec = []
#     # Position (normalized)
#     vec.append(pos / seq_length)
#     # WT amino acid (one-hot, 20)
#     wt_oh = [0.0] * 20
#     if wt_aa in aa_to_idx: wt_oh[aa_to_idx[wt_aa]] = 1.0
#     vec.extend(wt_oh)
#     # Mutant amino acid (one-hot, 20)
#     mt_oh = [0.0] * 20
#     if mut_aa in aa_to_idx: mt_oh[aa_to_idx[mut_aa]] = 1.0
#     vec.extend(mt_oh)

#     return np.array(vec, dtype=np.float32)

def encode_mutation(mut, seq_length=seq_length):
    wt_aa, pos, mut_aa = mut[0], int(mut[1:-1]), mut[-1]
    vec = []

    # Position (normalized)
    vec.append(pos / seq_length)

    # WT amino acid one-hot
    wt_oh = [0] * 20
    wt_oh[aa_to_idx[wt_aa]] = 1
    vec.extend(wt_oh)

    # Mutant amino acid one-hot
    mut_oh = [0] * 20
    mut_oh[aa_to_idx[mut_aa]] = 1
    vec.extend(mut_oh)

    # Tiny extra feature: same coarse biochemical group
    aa_group = {
        'A': 'hydrophobic', 'V': 'hydrophobic', 'I': 'hydrophobic', 'L': 'hydrophobic',
        'M': 'hydrophobic', 'F': 'aromatic', 'Y': 'aromatic', 'W': 'aromatic',
        'S': 'polar', 'T': 'polar', 'N': 'polar', 'Q': 'polar', 'C': 'polar', 'G': 'special', 'P': 'special',
        'D': 'negative', 'E': 'negative',
        'K': 'positive', 'R': 'positive', 'H': 'positive'
    }

    same_group = 1.0 if aa_group[wt_aa] == aa_group[mut_aa] else 0.0
    vec.append(same_group)

    return np.array(vec, dtype=np.float32)

class ProteinDataset(Dataset):
    def __init__(self, df, istrain=True):
        self.encodings = np.stack([encode_mutation(m) for m in df['mutant'].values])
        self.targets = df['DMS_score'].values.astype(np.float32) if istrain else np.zeros(len(df), dtype=np.float32)

    def __len__(self): return len(self.encodings)
    def __getitem__(self, idx): return torch.tensor(self.encodings[idx]), torch.tensor(self.targets[idx])

In [56]:
train_dataset = ProteinDataset(df_train)
test_dataset = ProteinDataset(df_test, istrain=False)

# split validation set
train_dataset, val_dataset = train_test_split(train_dataset, test_size=val_ratio, random_state=seed, shuffle=True)

# TODO: revise according to your own model
train_loader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

In [57]:
from sklearn.ensemble import GradientBoostingRegressor
from scipy.stats import spearmanr
import numpy as np

X_train = np.stack([x[0].numpy() for x in train_dataset])
y_train = np.array([x[1].numpy() for x in train_dataset])
X_val   = np.stack([x[0].numpy() for x in val_dataset])
y_val   = np.array([x[1].numpy() for x in val_dataset])
X_test  = np.stack([x[0].numpy() for x in test_dataset])

# Combine train and val for final training to get maximum data
X_full_train = np.concatenate([X_train, X_val])
y_full_train = np.concatenate([y_train, y_val])

# Train a Gradient Boosting Regressor and evaluate on validation set
gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
gbr.fit(X_train, y_train)

val_preds = gbr.predict(X_val)
val_spearman = spearmanr(y_val, val_preds)[0]
print(f"Gradient Boosting val Spearman: {val_spearman:.4f}")

# Retrain on full training data for final test predictions
gbr.fit(X_full_train, y_full_train)
y_test_pred = gbr.predict(X_test)
print("Predictions generated for test set.")

Gradient Boosting val Spearman: 0.4202
Predictions generated for test set.


XGBoost Model - SpearmanScore 0.25985

In [58]:
from xgboost import XGBRegressor
from scipy.stats import spearmanr
import numpy as np

X_train = np.stack([x[0].numpy() for x in train_dataset])
y_train = np.array([x[1].numpy() for x in train_dataset])

X_val = np.stack([x[0].numpy() for x in val_dataset])
y_val = np.array([x[1].numpy() for x in val_dataset])

X_test = np.stack([x[0].numpy() for x in test_dataset])

X_full_train = np.concatenate([X_train, X_val], axis=0)
y_full_train = np.concatenate([y_train, y_val], axis=0)

param_grid = [
    {"n_estimators": 400, "learning_rate": 0.03, "max_depth": 3, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1},
    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 3, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1},
    {"n_estimators": 600, "learning_rate": 0.03, "max_depth": 3, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1},

    {"n_estimators": 500, "learning_rate": 0.02, "max_depth": 3, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1},
    {"n_estimators": 700, "learning_rate": 0.02, "max_depth": 3, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1},

    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 2, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1},
    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 4, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1},

    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 3, "subsample": 0.9, "colsample_bytree": 0.8, "min_child_weight": 1},
    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 3, "subsample": 0.8, "colsample_bytree": 0.9, "min_child_weight": 1},

    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 3, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 3},
]

best_score = -1
best_params = None

for i, params in enumerate(param_grid, 1):
    model = XGBRegressor(
        objective='reg:squarederror',
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        min_child_weight=params["min_child_weight"],
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    val_preds = model.predict(X_val)
    score = spearmanr(y_val, val_preds)[0]

    print(f"[{i:02d}/{len(param_grid)}] params={params} -> val Spearman={score:.4f}")

    if score > best_score:
        best_score = score
        best_params = params

print("\nBest params:", best_params)
print("Best val Spearman:", round(best_score, 4))

final_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=best_params["n_estimators"],
    learning_rate=best_params["learning_rate"],
    max_depth=best_params["max_depth"],
    subsample=best_params["subsample"],
    colsample_bytree=best_params["colsample_bytree"],
    min_child_weight=best_params["min_child_weight"],
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_full_train, y_full_train)
final_test_preds = final_model.predict(X_test).astype(np.float32)

print("Test prediction range:",
      float(final_test_preds.min()),
      "to",
      float(final_test_preds.max()))
print("Predictions generated for test set.")

[01/10] params={'n_estimators': 400, 'learning_rate': 0.03, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1} -> val Spearman=0.4455
[02/10] params={'n_estimators': 500, 'learning_rate': 0.03, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1} -> val Spearman=0.4410
[03/10] params={'n_estimators': 600, 'learning_rate': 0.03, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1} -> val Spearman=0.4388
[04/10] params={'n_estimators': 500, 'learning_rate': 0.02, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1} -> val Spearman=0.4401
[05/10] params={'n_estimators': 700, 'learning_rate': 0.02, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1} -> val Spearman=0.4423
[06/10] params={'n_estimators': 500, 'learning_rate': 0.03, 'max_depth': 2, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1} -> val Spearman=0.4318
[07/10] pa

XGBoost Working model

In [59]:
from xgboost import XGBRegressor
from scipy.stats import spearmanr
import numpy as np

# Convert datasets to numpy arrays
X_train = np.stack([x[0].numpy() for x in train_dataset])
y_train = np.array([x[1].numpy() for x in train_dataset])

X_val = np.stack([x[0].numpy() for x in val_dataset])
y_val = np.array([x[1].numpy() for x in val_dataset])

X_test = np.stack([x[0].numpy() for x in test_dataset])

# Full training set for final refit
X_full_train = np.concatenate([X_train, X_val], axis=0)
y_full_train = np.concatenate([y_train, y_val], axis=0)

# Best XGBoost config found so far
model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    learning_rate=0.02,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    reg_lambda=1,
    reg_alpha=0,
    gamma=0,
    random_state=42,
    n_jobs=-1
)

# Train on train split for validation check
model.fit(X_train, y_train)

val_preds = model.predict(X_val)
val_spearman = spearmanr(y_val, val_preds)[0]

print(f"XGBoost val Spearman: {val_spearman:.4f}")
print(f"Validation prediction range: {val_preds.min():.6f} to {val_preds.max():.6f}")

# Refit on full train + val
final_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    learning_rate=0.02,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    reg_lambda=1,
    reg_alpha=0,
    gamma=0,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_full_train, y_full_train)

# Final test predictions
final_test_preds = final_model.predict(X_test).astype(np.float32)

print(f"Test prediction range: {final_test_preds.min():.6f} to {final_test_preds.max():.6f}")
print("Predictions generated for test set.")

XGBoost val Spearman: 0.4401
Validation prediction range: 0.037440 to 0.529361
Test prediction range: 0.024856 to 0.644932
Predictions generated for test set.


ETR Gradient Boost Model with spearman score 0.3266:

In [43]:
from sklearn.ensemble import ExtraTreesRegressor
from scipy.stats import spearmanr
import numpy as np

# Convert datasets to numpy arrays
X_train = np.stack([x[0].numpy() for x in train_dataset])
y_train = np.array([x[1].numpy() for x in train_dataset])

X_val = np.stack([x[0].numpy() for x in val_dataset])
y_val = np.array([x[1].numpy() for x in val_dataset])

X_test = np.stack([x[0].numpy() for x in test_dataset])

# Combine train + val for final training after validation check
X_full_train = np.concatenate([X_train, X_val], axis=0)
y_full_train = np.concatenate([y_train, y_val], axis=0)

# Train ExtraTrees on training split
etr = ExtraTreesRegressor(
    n_estimators=1200,
    max_depth=None,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

etr.fit(X_train, y_train)

# Validate
val_preds = etr.predict(X_val)
val_spearman = spearmanr(y_val, val_preds)[0]
print(f"ExtraTrees val Spearman: {val_spearman:.4f}")
print(f"Validation prediction range: {val_preds.min():.6f} to {val_preds.max():.6f}")

# Refit on full training data
etr.fit(X_full_train, y_full_train)

# Final test predictions
final_test_preds = etr.predict(X_test).astype(np.float32)
print(f"Test prediction range: {final_test_preds.min():.6f} to {final_test_preds.max():.6f}")
print("Predictions generated for test set.")

ExtraTrees val Spearman: 0.3266
Validation prediction range: 0.065172 to 0.582545
Test prediction range: 0.049180 to 0.580289
Predictions generated for test set.


Tried Ridge, GradientBoost, HistGradientBoost, and Extra Trees

In [44]:
from sklearn.ensemble import GradientBoostingRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import numpy as np

# Convert datasets to numpy arrays
X_train = np.stack([x[0].numpy() for x in train_dataset])
y_train = np.array([x[1].numpy() for x in train_dataset])

X_val = np.stack([x[0].numpy() for x in val_dataset])
y_val = np.array([x[1].numpy() for x in val_dataset])

X_test = np.stack([x[0].numpy() for x in test_dataset])

# Combine train + val for final training
X_full_train = np.concatenate([X_train, X_val], axis=0)
y_full_train = np.concatenate([y_train, y_val], axis=0)

# Scale only for Ridge
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
X_full_scaled = scaler.fit_transform(X_full_train)
X_test_full_scaled = scaler.transform(X_test)

models = {
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        random_state=42
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        learning_rate=0.03,
        max_iter=400,
        max_depth=6,
        min_samples_leaf=8,
        l2_regularization=0.2,
        random_state=42
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=800,
        max_depth=None,
        min_samples_leaf=3,
        max_features='sqrt',
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    ),
    "Ridge": Ridge(alpha=2.0)
}

val_scores = {}
trained_models = {}

# Train + evaluate
for name, model in models.items():
    if name == "Ridge":
        model.fit(X_train_scaled, y_train)
        val_preds = model.predict(X_val_scaled)
    else:
        model.fit(X_train, y_train)
        val_preds = model.predict(X_val)

    score = spearmanr(y_val, val_preds)[0]
    val_scores[name] = score
    trained_models[name] = model

    print(f"{name} val Spearman: {score:.4f}")

# Pick best model
best_model_name = max(val_scores, key=val_scores.get)
print(f"\nBest model on validation: {best_model_name} ({val_scores[best_model_name]:.4f})")

# Refit best model on full train + val
if best_model_name == "Ridge":
    best_model = Ridge(alpha=2.0)
    best_model.fit(X_full_scaled, y_full_train)
    final_test_preds = best_model.predict(X_test_full_scaled).astype(np.float32)
else:
    best_model = models[best_model_name]
    best_model.fit(X_full_train, y_full_train)
    final_test_preds = best_model.predict(X_test).astype(np.float32)

print(f"Test prediction range: {final_test_preds.min():.6f} to {final_test_preds.max():.6f}")
print("Predictions generated for test set.")

GradientBoosting val Spearman: 0.4159
HistGradientBoosting val Spearman: 0.3826
ExtraTrees val Spearman: 0.3434
Ridge val Spearman: 0.2011

Best model on validation: GradientBoosting (0.4159)
Test prediction range: 0.037534 to 0.754732
Predictions generated for test set.


Neural Network attempt

In [45]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from scipy.stats import spearmanr
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Convert datasets into loaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

input_dim = train_dataset[0][0].shape[0]

class MLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.30),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.25),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

model = MLPRegressor(input_dim).to(device)

criterion = nn.SmoothL1Loss()   # Huber-style loss, usually stabler than plain MSE
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=4
)

def evaluate_spearman(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device).float()
            yb = yb.to(device).float()
            out = model(xb)
            preds.append(out.cpu().numpy())
            targets.append(yb.cpu().numpy())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    return spearmanr(targets, preds)[0], preds, targets

best_model_state = None
best_val_spearman = -1.0
patience = 10
patience_counter = 0
num_epochs = 60

for epoch in range(num_epochs):
    model.train()
    train_losses = []

    for xb, yb in train_loader:
        xb = xb.to(device).float()
        yb = yb.to(device).float()

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    val_spearman, val_preds, val_targets = evaluate_spearman(model, val_loader)
    scheduler.step(val_spearman)

    avg_train_loss = np.mean(train_losses)
    print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Spearman: {val_spearman:.4f}")

    if val_spearman > best_val_spearman:
        best_val_spearman = val_spearman
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping triggered.")
        break

print(f"\nBest validation Spearman: {best_val_spearman:.4f}")

# Load best model
model.load_state_dict(best_model_state)

# Predict on test set
model.eval()
test_preds = []

with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device).float()
        out = model(xb)
        test_preds.append(out.cpu().numpy())

final_test_preds = np.concatenate(test_preds).astype(np.float32)

print(f"Test prediction range: {final_test_preds.min():.6f} to {final_test_preds.max():.6f}")
print("Predictions generated for test set.")

Using device: cuda
Epoch 01 | Train Loss: 0.0425 | Val Spearman: 0.1133
Epoch 02 | Train Loss: 0.0324 | Val Spearman: 0.0765
Epoch 03 | Train Loss: 0.0275 | Val Spearman: 0.0825
Epoch 04 | Train Loss: 0.0240 | Val Spearman: 0.0639
Epoch 05 | Train Loss: 0.0245 | Val Spearman: 0.0939
Epoch 06 | Train Loss: 0.0236 | Val Spearman: 0.1325
Epoch 07 | Train Loss: 0.0252 | Val Spearman: 0.1491
Epoch 08 | Train Loss: 0.0196 | Val Spearman: 0.1919
Epoch 09 | Train Loss: 0.0192 | Val Spearman: 0.2238
Epoch 10 | Train Loss: 0.0219 | Val Spearman: 0.2293
Epoch 11 | Train Loss: 0.0204 | Val Spearman: 0.2184
Epoch 12 | Train Loss: 0.0192 | Val Spearman: 0.2272
Epoch 13 | Train Loss: 0.0189 | Val Spearman: 0.2210
Epoch 14 | Train Loss: 0.0182 | Val Spearman: 0.2461
Epoch 15 | Train Loss: 0.0183 | Val Spearman: 0.2572
Epoch 16 | Train Loss: 0.0186 | Val Spearman: 0.2638
Epoch 17 | Train Loss: 0.0201 | Val Spearman: 0.2842
Epoch 18 | Train Loss: 0.0172 | Val Spearman: 0.2572
Epoch 19 | Train Loss: 0.01

To view the head and create predictions.csv

In [62]:
df_test['DMS_score_predicted'] = final_test_preds
df_test

,mutant,sequence,DMS_score_predicted,id
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.330485,0
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.293583,1
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.315623,2
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.382560,3
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.287683,4
...,...,...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.393645,11319
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.431231,11320
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.362711,11321
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.490995,11322


In [63]:
df_test['id'] = df_test.index
df_test[['id', 'DMS_score_predicted']].rename(columns={'DMS_score_predicted': 'DMS_score'}).to_csv('predictions.csv', index=False)
# df_test[['mutant', 'DMS_score_predicted']].to_csv('predictions.csv', index=False)

## 3. Select query for next round

In [22]:
df_test.sort_values('DMS_score_predicted', ascending=False).head(100)

,mutant,sequence,DMS_score_predicted,id
6223,R387K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11307.333008,6223
7894,R475K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11307.000000,7894
7311,R444K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11305.333008,7311
10143,R593K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11302.333008,10143
2403,R133K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11292.000000,2403
...,...,...,...,...
6980,E427D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11036.000000,6980
5145,E325D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11035.333008,5145
1409,I76V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11034.333008,1409
3569,E207D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11034.000000,3569


In [23]:
best_mutants = df_test.sort_values('DMS_score_predicted', ascending=False).head(10)['mutant'].values
with open('top10.txt', 'w') as f:
    for mut in best_mutants:
        f.write(mut + '\n')
df_test.sort_values('DMS_score_predicted', ascending=False).head(100)

,mutant,sequence,DMS_score_predicted,id
6223,R387K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11307.333008,6223
7894,R475K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11307.000000,7894
7311,R444K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11305.333008,7311
10143,R593K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11302.333008,10143
2403,R133K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11292.000000,2403
...,...,...,...,...
6980,E427D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11036.000000,6980
5145,E325D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11035.333008,5145
1409,I76V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11034.333008,1409
3569,E207D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,11034.000000,3569


In [24]:
# Example: randomly select 100 test variants to be queried.
# Note: random selection may not be a good strategy
# TODO: select query mutants for the next round based on your own criteria

querys = np.random.choice(df_test.mutant.values, size=100, replace=False)
querys


array(['G170T', 'E398V', 'N7I', 'E415G', 'S492R', 'Q570C', 'V217W',
       'P394I', 'L147K', 'H140K', 'E379Y', 'L313Y', 'P342D', 'A219R',
       'P354H', 'D628K', 'F256P', 'W582A', 'S439A', 'D575V', 'T114D',
       'I286R', 'R605A', 'F230W', 'A460T', 'L627K', 'H142M', 'T425C',
       'A649R', 'D162T', 'K467Q', 'F396N', 'W94S', 'L61Y', 'A586K',
       'D244C', 'L213D', 'V430W', 'L479H', 'F236G', 'L436H', 'W594P',
       'G312T', 'G421D', 'L620I', 'R357L', 'K418E', 'E636F', 'R362I',
       'P555Y', 'D628P', 'K333E', 'P12V', 'Q526C', 'F581Y', 'S32H',
       'M153F', 'F587Q', 'P551R', 'I496K', 'N443S', 'E122I', 'D575I',
       'F63P', 'D37S', 'Q476A', 'V542H', 'R369V', 'E46W', 'S102G',
       'N488P', 'S25H', 'V542P', 'E428H', 'R5L', 'S464D', 'E653Y',
       'L125P', 'Q146T', 'P458E', 'N541W', 'P458F', 'R593Y', 'E457Q',
       'R81K', 'D288W', 'G86P', 'F238E', 'F625R', 'L620R', 'P485T',
       'A241L', 'S391R', 'N643S', 'F507V', 'T646M', 'K179V', 'S24E',
       'C451W', 'Q526K'], dtype=obj

In [25]:
with open('query.txt', 'w') as f:
  for mutant in querys:
    f.write(mutant+'\n')